This notebook accompanies [08-neural-quantum-states](https://xing-phys.github.io/ai-for-physics/08-neural-quantum-states.html). Run cells from top to bottom. The web chapter provides typeset citations and numbered cross-references. Teaching code: MIT; text and figures: CC BY 4.0. All data are generated locally.


# Neural quantum states: learning amplitudes, signs, and correlations 

A magnetic ground state may be a coherent superposition of billions of spin configurations. We want a compact rule that returns the amplitude of any requested configuration and can adapt when the Hamiltonian changes. A neural quantum state (NQS) supplies such a rule. Its inputs are particle coordinates, spin labels, or occupation numbers; its output describes a wavefunction. The network is a variational *ansatz*, meaning a chosen family of trial states, rather than an extra physical law.

The useful question is not whether a network can interpolate a table. It is whether its architecture captures the correlations and phase structure of a physical problem, while allowing affordable sampling and observable evaluation. We will build two complementary designs: an analytically marginalized hidden-variable model, and an autoregressive state whose sampling probabilities are normalized by construction.

::: {.callout-note title="Learning goals and prerequisites"}
Use [Quantum foundations](07-quantum-foundations.ipynb) for Hilbert spaces, observables, and entanglement. A neural network is a composition of affine maps and nonlinear functions, with adjustable coefficients called parameters; derivatives of this composition can be obtained by the chain rule. We explain each architecture needed here. You should finish able to derive an RBM amplitude, construct a normalized autoregressive wavefunction, identify what a phase network contributes, and enforce a simple symmetry. [Variational Monte Carlo](09-variational-monte-carlo.ipynb) will derive the training objective and optimizer.
:::

## What should a network output? 

For a spin configuration $s=(s_1,\ldots,s_N)$, a convenient output is

$$
f_\theta(s)=u_\theta(s)+i v_\theta(s),\qquad
\widetilde\psi_\theta(s)=e^{f_\theta(s)}.
$$ 

Here $u$ determines the logarithmic magnitude and $v$ the phase. The tilde reminds us that the state may be unnormalized. The Born distribution is

$$
p_\theta(s)=\frac{e^{2u_\theta(s)}}{Z_\theta},\qquad
Z_\theta=\sum_s e^{2u_\theta(s)}.
$$

The factor of two is essential: a probability is an amplitude magnitude squared. The phase does not affect sampling in this basis, but it affects off-diagonal observables through

$$
\frac{\widetilde\psi(s')}{\widetilde\psi(s)}
=\exp\{u(s')-u(s)+i[v(s')-v(s)]\}.
$$

Computing differences of logarithms avoids multiplying many very small amplitudes. For real positive sums, the log-sum-exp identity is helpful: $\log\sum_j e^{a_j}=m+\log\sum_je^{a_j-m}$ with $m=\max_j a_j$. Complex sums require additional care because cancellation can create exact zeros.

A finite exponential never vanishes, so a bare log-amplitude network cannot represent exact nodes at finite outputs. For a constrained Hilbert space, we can explicitly set forbidden configurations to zero. For fermions, determinants naturally produce nodes. If we need exact zeros from interference, a sum of amplitudes or another architecture may be preferable. This is a representational issue, not merely a numerical nuisance: a fixed wrong nodal structure can exclude the desired physics.

One overall additive real constant in $u$ changes only normalization, and one overall constant in $v$ changes only global phase. These are redundant directions in parameter space. They do not change the normalized quantum state. Later they will appear as zero modes of the variational metric.

## Restricted Boltzmann machines, derived by summing hidden spins 

Introduce visible spins $s_i=\pm1$ and auxiliary hidden variables $h_j=\pm1$, with $j=1,\ldots,M$. A restricted Boltzmann machine (RBM) has connections between visible and hidden variables, but no hidden–hidden or visible–visible connections. For a wavefunction, define

$$
\widetilde\psi_\theta(s)=\sum_{h_1=\pm1}\cdots\sum_{h_M=\pm1}
\exp\left(\sum_i a_i s_i+\sum_j b_jh_j+\sum_{ij}W_{ij}s_ih_j\right).
$$ 

The coefficients may be complex. In that case the expression is an amplitude model, not a positive classical joint probability. For fixed $s$, all terms involving a given $h_j$ combine into $h_jB_j(s)$, where $B_j=b_j+\sum_iW_{ij}s_i$. Since the hidden spins do not couple, their sum factorizes:

$$
\begin{aligned}
\widetilde\psi_\theta(s)
&=e^{\sum_i a_i s_i}\prod_j\sum_{h_j=\pm1}e^{h_jB_j(s)}\\
&=e^{\sum_i a_i s_i}\prod_j2\cosh B_j(s).
\end{aligned}
$$ 

The sum over $2^M$ hidden configurations is evaluated in order $NM$ arithmetic operations. Hidden variables can induce correlations between many visible spins because one hidden spin can connect to many sites. The network stores $N+M+NM$ coefficients, not an exponentially large amplitude table. This alone does not imply that a fixed $M$ accurately represents every ground state.

For a real positive RBM with no zeros, logarithmic derivatives are particularly simple:

$$
\frac{\partial\log\widetilde\psi}{\partial a_i}=s_i,\quad
\frac{\partial\log\widetilde\psi}{\partial b_j}=\tanh B_j,\quad
\frac{\partial\log\widetilde\psi}{\partial W_{ij}}=s_i\tanh B_j.
$$ 

These follow by differentiating $\log\cosh B$, using $(d/dB)\log\cosh B=\tanh B$, and applying the chain rule. Complex parameters require a specified complex-derivative convention; alternatively split their real and imaginary parts and use ordinary real derivatives. We use the latter general convention in the next chapter.

### Worked representation of an Ising correlation

Take two visible spins and one hidden spin, with $a_i=b=0$ and $W_{11}=W_{21}=w$ real. Then $\widetilde\psi(s_1,s_2)=2\cosh[w(s_1+s_2)]$. The amplitudes of parallel spins are $2\cosh(2w)$, and those of antiparallel spins are $2$. Their *probability* ratio is $\cosh^2(2w)$, not $\cosh(2w)$.

Suppose we want a positive Jastrow state $\psi_K(s)\propto e^{Ks_1s_2/2}$ with $K\geq0$. Its parallel/antiparallel amplitude ratio is $e^K$. Choose $\cosh(2w)=e^K$, or $w=\frac12\operatorname{arcosh}(e^K)$. Up to one common normalization factor, the single hidden spin reproduces the desired state exactly. The Born distribution is the two-spin classical Ising distribution proportional to $e^{Ks_1s_2}$. This example explains how hidden units can encode a physical correlation. It does not establish an efficient representation for arbitrary frustrated sign patterns.

When the Hamiltonian has nonpositive off-diagonal matrix elements in a connected basis, its lowest state can be chosen with nonnegative coefficients. One way to see the energy advantage is to replace coefficients by their magnitudes: diagonal contributions are unchanged, while each nonpositive real off-diagonal contribution cannot increase. Irreducibility then permits a strictly positive ground vector. Such Hamiltonians are called *stoquastic in that basis*. The qualifier matters: a basis change can alter the sign pattern. For these ground states a real positive ansatz may be justified. Excited states, real-time dynamics, and many frustrated or fermionic problems generally require signs or complex phases.

## Autoregressive states: normalization by the probability chain rule 

An alternative begins with the exact probability identity

$$
p(s_1,\ldots,s_N)=p(s_1)\prod_{i=2}^Np(s_i\mid s_1,\ldots,s_{i-1}).
$$ 

Each factor is a conditional probability for the next spin, given the preceding ones in a chosen ordering. A network returns two real logits $\ell_i(+1),\ell_i(-1)$ and converts them into probabilities using

$$
p_\theta(s_i=a\mid s_{<i})=
\frac{e^{\ell_i(a;s_{<i})}}{e^{\ell_i(+1;s_{<i})}+e^{\ell_i(-1;s_{<i})}}.
$$

The softmax denominator normalizes each two-outcome distribution. The full distribution is normalized because summing over $s_N$ removes the last factor, summing over $s_{N-1}$ removes the preceding one, and continuing leaves $\sum_{s_1}p(s_1)=1$. No exponential normalization sum is needed.

A normalized wavefunction is then

$$
\psi_\theta(s)=\sqrt{p_\theta(s)}\,e^{i\phi_\theta(s)}.
$$ 

The phase $\phi$ can be a separate scalar network or a sum of conditional phase contributions. It need not itself be a probability. Sampling proceeds in the same order: draw $s_1$, evaluate the next conditional using that value, draw $s_2$, and continue. Repeating the process with fresh random numbers produces independent configurations from the modeled distribution. This eliminates Markov-chain burn-in and serial autocorrelation between samples; it does not eliminate Monte Carlo variance, rare-event difficulties, or the cost of sequential conditional evaluations.

A recurrent network implements the conditionals through a memory vector. Starting from $h_0$, update $h_i=\tanh(Ah_{i-1}+Bs_i+c)$ and generate the logits for $s_{i+1}$ from $h_i$. The vector summarizes the preceding spins. A transformer instead lets a site combine information from earlier sites by weighted attention. Its *causal mask* forbids access to future spins when calculating a conditional. Without this restriction, multiplying the outputs is generally not the claimed normalized chain rule. A masked feed-forward network supplies a third implementation. The physical construction is the same; the mechanisms for summarizing the prefix differ.

### Worked GHZ state and a phase that sampling cannot see

For $|\mathrm{GHZ}\rangle=(|\uparrow\cdots\uparrow\rangle+|\downarrow\cdots\downarrow\rangle)/\sqrt2$, set $p(s_1=\pm1)=1/2$ and every subsequent spin equal to $s_1$ with conditional probability one. These deterministic conditionals are most naturally implemented by masks; finite logits only approximate probability one. Set the phase to zero on the two supported configurations. The state is exactly normalized and has entropy $\log2$ across any nontrivial bipartition, even though it has only two nonzero amplitudes.

Changing the phase of the all-down configuration by $\pi$ gives the odd GHZ state. The samples do not change at all. However $\langle\prod_i\sigma_i^x\rangle$ changes from $+1$ to $-1$. The amplitude and phase networks therefore solve logically distinct parts of the representation problem. If the Hamiltonian probes phase through long-range interference, fitting a sampling distribution accurately may still leave the energy or dynamics wrong.

## Constraints and symmetry are physical information 

Suppose total $z$ magnetization is fixed, equivalently the number of up spins is fixed at $n_\uparrow$. An autoregressive sampler can keep track of the remaining number. A proposed choice is masked out if it would exceed $n_\uparrow$ or make it impossible to reach the target with the remaining sites. Renormalize the two allowed conditional outcomes after masking. The sequential normalization proof still applies inside the allowed configuration set. This is more reliable than adding a penalty and hoping the network learns a conservation law.

For a finite symmetry group $G$, a group element acts by permuting configurations. To project into a one-dimensional representation with character $\chi(g)$, define

$$
\psi_\chi(s)=\sum_{g\in G}\chi(g)^*\psi_\theta(g^{-1}s).
$$ 

To check its transformation, substitute $s\mapsto hs$ and relabel $g=hk$. Since $\chi(hk)^*=\chi(h)^*\chi(k)^*$, the result is $\psi_\chi(hs)=\chi(h)^*\psi_\chi(s)$ under this configuration convention. The associated ket carries character $\chi(h)$ when the operator acts as $U_h|s\rangle=|hs\rangle$. Writing the convention removes an otherwise easy complex-conjugation ambiguity. For the invariant sector, $\chi=1$ and these distinctions disappear.

Projection costs up to $|G|$ network evaluations and can create nodes by cancellation. It can also destroy the direct autoregressive factorization: the squared magnitude of a sum of amplitudes is not the average of their squared magnitudes. A mixture of transformed probability distributions is easy to sample, but it is generally a different state. A symmetry-aware architecture that is equivariant at intermediate layers can reduce this cost, although its exact output representation must still be checked.

Symmetry is not always something to impose indiscriminately. A finite system can have a symmetric ground state while a thermodynamic phase admits broken-symmetry states. A chosen momentum or spin sector may exclude a competing state. Compare sectors and finite-size scaling instead of treating one symmetry choice as proof of the phase.

## A small executable amplitude laboratory 

This example enumerates a small RBM, verifies its analytic derivatives, and compares parallel-spin probabilities with the formula above. Enumeration is deliberate: it gives a trusted reference before replacing sums by sampling.

In [1]:
import itertools
import numpy as np
states = np.array(list(itertools.product([-1., 1.], repeat=2)))
K = 0.8
w = 0.5 * np.arccosh(np.exp(K))
def log_amplitude(weight):
    return np.log(2.0 * np.cosh(weight * states.sum(axis=1)))
logpsi = log_amplitude(w)
weights = np.exp(2 * (logpsi - logpsi.max()))
p = weights / weights.sum()
parallel = states[:, 0] == states[:, 1]
assert np.allclose(p[parallel][0] / p[~parallel][0], np.exp(2*K))
eps = 1e-6
finite_difference = (log_amplitude(w+eps)-log_amplitude(w-eps))/(2*eps)
analytic = states.sum(axis=1) * np.tanh(w * states.sum(axis=1))
assert np.allclose(finite_difference, analytic, atol=1e-8)
correlation = np.sum(p * states[:, 0] * states[:, 1])
assert np.allclose(correlation, np.tanh(K))
print(f"P(parallel): {p[parallel].sum():.8f}")
print(f"<s1 s2>: {correlation:.8f}; analytic: {np.tanh(K):.8f}")

P(parallel): 0.83201839
<s1 s2>: 0.66403677; analytic: 0.66403677


The correlation follows independently from the partition function $Z=4\cosh K$: $\partial_K\log Z=\tanh K$. This cross-check compares the neural representation with a physical calculation, not merely two implementations of the same network formula.

## Expressivity, optimization, and scientific use 

Three different questions are often conflated. *Expressivity* asks whether good parameters exist. *Optimization* asks whether a practical training process finds them. *Computation* asks whether samples, derivatives, and observables remain affordable at those parameters. A mathematical universal-approximation statement addresses the first question in an appropriate limit. It need not supply polynomial parameter counts, a constructive training method, or bounded sampling variance. Increasing network width may improve one bottleneck while worsening another.

RBM states established a useful bridge between statistical learning and variational quantum simulation [@q-carleo2017]. Recurrent autoregressive wavefunctions provide a different bridge: exact normalization and direct samples are built into the representation [@q-hibatallah2020]. Transformer quantum states can condition on Hamiltonian parameters to represent families of problems [@q-zhang2023]. Their reported transfer behavior is empirical and task dependent. The original work has an erratum correcting plotted fit exponents [@q-zhangerratum2025]; one should consult it before using those scaling estimates. Conditioning does not ensure extrapolation across a phase transition or into a new system size.

Recent architecture research also asks whether familiar physical operations can guide the network. One 2026 study interprets layer updates through a latent imaginary-time construction and tests weight sharing and propagation-inspired designs in frustrated magnets [@q-yamazaki2026]. This is an architectural interpretation and numerical comparison, not a theorem that a network's hidden variables are physical imaginary-time states. A different approach changes the computational basis around a physically motivated reference and learns its corrections [@q-sobral2025]. These examples suggest two independent design choices: the map that processes configurations and the basis in which the amplitudes are represented. Both should be compared at controlled computational cost.

A physically revealing application is the comparison of candidate phases in Rydberg arrays. The recurrent-network study of kagome Rydberg models by Hibat-Allah and collaborators questions certain exotic-phase interpretations and emphasizes sampling autocorrelations [@q-rydberg2025]. This is an example of neural representations contributing to scientific *discrimination*. Its finite-size and variational evidence should be weighed alongside competing solvers and observables, rather than promoted to a theorem excluding every spin liquid.

The 2026 perspective [@q-rigo2026] frames sign structure and reliable bias control as continuing challenges. A useful independent project could keep the architecture fixed while varying the physical basis, symmetry sector, and sampling method. If the predicted phase changes under these numerical choices more than under the physical tuning parameter, the calculation has diagnosed its own uncertainty. That outcome is scientifically useful even when it does not produce the desired discovery.

## Exercises and selected solutions 

1. Derive the RBM derivatives in @eq-quantum-rbm-derivatives without using automatic differentiation. What singularities can appear for complex $B_j$?
2. For a three-spin autoregressive model, write all eight probabilities and verify their sum is one by successive conditional sums.
3. Construct an exact autoregressive distribution for a uniform state with precisely two up spins on four sites. Explain how each conditional depends on the number of remaining slots.
4. Show explicitly that a symmetric mixture $[p(s)+p(gs)]/2$ is generally not the Born distribution of $\psi(s)+\psi(gs)$.
5. Replace the phase of one basis amplitude in the two-spin RBM laboratory by $\pi$. Calculate a transverse correlation before and after, and verify that every diagonal $z$ observable is unchanged.

::: {.callout-tip title="Worked solutions"}
For exercise 3, suppose $r$ up spins remain to be placed among $m$ remaining sites. The uniform fixed-count distribution uses $p(\uparrow)=r/m$ and $p(\downarrow)=(m-r)/m$, followed by the appropriate updated count. A configuration with two up spins receives probability $(2!2!)/4!=1/6$, independent of their positions. At $r=0$ or $r=m$, the choice is deterministic. No rejection step is needed.

For exercise 4, $|\psi(s)+\psi(gs)|^2=|\psi(s)|^2+|\psi(gs)|^2+2\operatorname{Re}[\psi(s)^*\psi(gs)]$. The interference term is absent from the mixture. For the amplitudes $\psi(s)=1$ and $\psi(gs)=-1$, the projected amplitude vanishes even though both mixture contributions are positive. Projection and classical mixing represent different physical operations.
:::